In [1]:
import pandas as pd
import numpy as np
import sqlite3

df = pd.read_csv(r"C:\Users\jayal\Documents\Marketing_Attribution_Project\Cleaned_attribution_data.csv")
print(df.shape)
df.head()

(586737, 6)


,cookie,time,interaction,conversion,conversion_value,channel
0,00000FkCnDfDDf0iC97iC703B,2018-07-03T13:02:11Z,impression,0,0.0,Instagram
1,00000FkCnDfDDf0iC97iC703B,2018-07-17T19:15:07Z,impression,0,0.0,Online Display
2,00000FkCnDfDDf0iC97iC703B,2018-07-24T15:51:46Z,impression,0,0.0,Online Display
3,00000FkCnDfDDf0iC97iC703B,2018-07-29T07:44:51Z,impression,0,0.0,Online Display
4,0000nACkD9nFkBBDECD3ki00E,2018-07-03T09:44:57Z,impression,0,0.0,Paid Search


In [ ]:
#June 22,2026
# Note: Ad spend and clicks data is simulated for analysis purposes
# as actual advertising platform data was not available for this dataset

In [2]:
#Simulated Ad Spend Data 
ad_spend_data = {
    'channel': ['Facebook','Instagram','Online Video','Online Display','Paid Search'],
    'spend': [12000,8000,6000,4000,15000],
    'clicks': [7200,5100,3800,2900,8500]
}
ad_spend = pd.DataFrame(ad_spend_data)
ad_spend.to_csv('ad_spend.csv',index=False)
print("ad_spend.csv created")
ad_spend

ad_spend.csv created


,channel,spend,clicks
0,Facebook,12000,7200
1,Instagram,8000,5100
2,Online Video,6000,3800
3,Online Display,4000,2900
4,Paid Search,15000,8500


In [4]:
import os
print(os.getcwd())

C:\Users\jayal


In [ ]:
#June 25th 2026

In [7]:
import pandas as pd
import numpy as np
import sqlite3

In [8]:
#KPI Calculation
channel_stats = df.groupby('channel').agg(conversions = ('conversion','sum'),
                                         revenue = ('conversion_value','sum')
                                         ).reset_index()

In [9]:
#Merge with ad spend
kpi = pd.merge(channel_stats,ad_spend,on = 'channel',how = 'left')

In [10]:
#KPI calculations
kpi['CPC'] = (kpi['spend']/kpi['clicks']).round(2)
kpi['CAC'] = (kpi['spend']/kpi['conversions']).round(2)
kpi['ROAS'] = (kpi['revenue']/kpi['spend']).round(2)

kpi = kpi.replace([np.inf, -np.inf],0)
print("KPI Calculated!")
kpi

KPI Calculated!


,channel,conversions,revenue,spend,clicks,CPC,CAC,ROAS
0,Facebook,5301,33143.5,12000,7200,1.67,2.26,2.76
1,Instagram,2244,14039.5,8000,5100,1.57,3.57,1.75
2,Online Display,2139,13298.5,4000,2900,1.38,1.87,3.32
3,Online Video,3408,21418.0,6000,3800,1.58,1.76,3.57
4,Paid Search,4547,28331.5,15000,8500,1.76,3.30,1.89


In [ ]:
#June 26,2026

In [3]:
#star schema
import sqlite3
conn = sqlite3.connect('marketing_attribution.db')

In [4]:
#dim_channel
dim_channel = ad_spend[['channel']].copy()
dim_channel['channel_id'] =  range(1,len(dim_channel)+1)
dim_channel = dim_channel[['channel_id','channel']]
dim_channel.to_sql('dim_channel',conn,if_exists='replace',index=False)
print("dim channel created!")
dim_channel

dim channel created!


,channel_id,channel
0,1,Facebook
1,2,Instagram
2,3,Online Video
3,4,Online Display
4,5,Paid Search


In [6]:
#dim_date
df['time'] = pd.to_datetime(df['time'])
dim_date = df[['time']].drop_duplicates().copy()
dim_date['date_id'] = range(1,len(dim_date)+1)
dim_date['date'] = dim_date['time'].dt.date
dim_date['month'] = dim_date['time'].dt.month
dim_date['year'] = dim_date['time'].dt.year
dim_date = dim_date[['date_id','date','month','year']]

dim_date.to_sql('dim_date',conn,if_exists='replace',index=False)
print("dim_date created!")
dim_date.head()


dim_date created!


,date_id,date,month,year
0,1,2018-07-03,7,2018
1,2,2018-07-17,7,2018
2,3,2018-07-24,7,2018
3,4,2018-07-29,7,2018
4,5,2018-07-03,7,2018


In [8]:
#dim_user
dim_user = df[['cookie']].drop_duplicates().copy()
dim_user.columns = ['user_id']

dim_user.to_sql('dim_user',conn,if_exists='replace',index=False)
print("dim_user created!")
print(f"Total unique users:{len(dim_user)}")

dim_user created!
Total unique users:240108


In [11]:
#dim_interaction
dim_interaction = df[['interaction']].drop_duplicates().copy()
dim_interaction['interaction_id'] = range(1,len(dim_interaction)+1)
dim_interaction = dim_interaction[['interaction_id','interaction']]
dim_interaction.to_sql('dim_interaction',conn,if_exists='replace',index=False)
print("Dim interaction created!")
dim_interaction

Dim interaction created!


,interaction_id,interaction
0,1,impression
22,2,conversion


In [13]:
#fact table
fact = df[df['conversion']==1].copy()
fact = fact.merge(dim_channel, on='channel',how='left')
fact = fact.merge(dim_interaction,on='interaction',how='left')
fact = fact.rename(columns={'cookie':'user_id'})
fact_conversions = fact[['user_id','channel_id','interaction_id','time','conversion_value']].copy()
fact_conversions.columns = ['user_id','channel_id','interaction_id','conversion_date','revenue']

fact_conversions.to_sql('fact_conversions',conn,if_exists='replace',index=False)
print("Fact conversions created")
print(f"Total conversion records:{len(fact_conversions)}")
fact_conversions.head()

Fact conversions created
Total conversion records:17639


,user_id,channel_id,interaction_id,conversion_date,revenue
0,0007oEBhnoF97AoEE3BCkFnhB,5,2,2018-07-06 13:45:29+00:00,6.5
1,00090n9EBBEkA000C7Cik999D,1,2,2018-07-05 06:53:53+00:00,8.0
2,000h3n9nC0hFhE3CCnkkAof7n,1,2,2018-07-19 14:31:57+00:00,6.0
3,000hCBnCB7oi7ADAEnEBCnBEE,3,2,2018-07-25 11:15:16+00:00,6.5
4,000kiDB3D0fCfDAohCDB3ohko,1,2,2018-07-26 16:16:21+00:00,7.5
